#**Josh Updated RFDiffusion Version 4 - Incremental Saving**

RFDiffusion Interface Resurfacing Pipeline
==========================================

A generalized pipeline for protein interface resurfacing using:
- RFDiffusion for backbone generation
- ProteinMPNN for sequence design  
- AlphaFold2-Multimer for structure validation

**NEW IN V4**: Incremental saving to Google Drive after each design validation.
Results are saved as they're generated, so even if the session times out,
you'll have partial results.

USAGE:
1. Fill in all parameters in Cell 0 (Configuration)
2. Upload your template PDB when prompted in Cell 1
3. Run all remaining cells

Author: Josh W

In [ ]:
#@title Cell 0: Configuration (EDIT THIS CELL)
#@markdown ## Project Settings
project_name = "Sr50_Pwl2_MLA3_binding_interface"  #@param {type:"string"}

#@markdown ## PDB Source Settings
#@markdown Choose ONE of the following options:
pdb_source = "custom"  #@param ["pdb_database", "custom"]
#@markdown - **pdb_database**: Downloads from RCSB PDB (use 4-letter code)
#@markdown - **custom**: Uses uploaded/Drive file (AlphaFold predictions, etc.)

pdb_id = "Sr50_Pwl2_AF2"  #@param {type:"string"}
#@markdown - **For pdb_database**: use 4-letter PDB code (e.g., "7XC2")
#@markdown - **For custom**: use descriptive name (becomes filename, e.g., "Sr50_Pwl2_AF2" → sr50_pwl2_af2.pdb)

#@markdown ### Custom PDB Path (optional)
#@markdown If you have the file on Google Drive, specify the full path:
custom_pdb_drive_path = ""  #@param {type:"string"}
#@markdown Leave empty to upload via browser or use default Drive folder location

#@markdown ## Structure Information
receptor_chain = "A"  #@param {type:"string"}
effector_chain = "B"  #@param {type:"string"}

#@markdown ### Structure Sequences
#@markdown These MUST match the actual residues in your PDB file
structure_receptor = "DGTGNSISSQSNVRRLSLQNKMEDHQAKPLTNIMSMSRLRSITIFPPAVSIMPSLSMFKVLRVLDLSNCDLGKSSSLQLNLKGVGHLIHLRYLGLANTKISELPTDIGNLQFLEVLDLENNYKLDELPSTLFKLRRLIYLNVWLNKVVPTLGVLQNLTSIEVLRGVLVSLNIIAQELGKLSRLRDLQISFKDDSLDLYESFVKSLCNLHQIESLCIVCNSKETSSELMDLLGEQWVPPVHLRIFVSYMPSQLSALRGWIKRDPSHLSNLSELILWPVKEVQQEDLEIIGGLLSLHHLWIKSTHQTQRLLVIPADGFRCMVEFKLDCGSAAQIMFEPGALPRAEAVAFSLGVRVAKDDGNCRFDLGLQGNLLSLRRQVYVWMYCGGARVGEAKEAEAAVRHALKADPHVYYISIQMRPRIAKGAHDDDLCEEWTDF"  #@param {type:"string"}
structure_effector = "GGWTNKQFYNDKGEREGSISIRKGSEGDFNYGPSYPGGPDRMVRVHENNGNIRGMPPGYSLGPDHQEDKSDRQYYNRHGYHVGDGPAEYGNHGGGQWGDGYYGPPGEFTHEHREQREEGCNIM"  #@param {type:"string"}

#@markdown ## Receptor start position (e.g. if you need to select just one domain)
receptor_start_pdb = 1  #@param {type:"integer"}

#@markdown ## RFDiffusion Contig String
#@markdown Format: "EffectorChain LRRstart-end/1-1/LRRstart-end/..."
contigs = "B A1-400/2-2/A403/21-21/A426-435"  #@param {type:"string"}

#@markdown ## Hotspot Residues
#@markdown Effector residues near the interface (format: "B123,B124,B125")
hotspot = "B24,B25,B26,B27,B28,B29,B40,B41,B42,B43,B44,B45,B46,B47,B85,B86,B87,B88,B89,B90,B91,B92,B93,B94,B95"  #@param {type:"string"}

#@markdown ## RFDiffusion Parameters
num_designs = 32  #@param ["1", "2", "4", "8", "16", "32"] {type:"raw"}
iterations = 100  #@param ["25", "50", "100", "150", "200"] {type:"raw"}
visual = "image"  #@param ["none", "image", "interactive"]

#@markdown ## Symmetry Settings
symmetry = "none"  #@param ["none", "auto", "cyclic", "dihedral"]
order = 1  #@param {type:"raw"}
add_potential = True  #@param {type:"boolean"}

#@markdown ## MPNN Parameters
num_seqs = 2  #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
mpnn_sampling_temp = 0.25  #@param {type:"raw"}
rm_aa = "C"  #@param {type:"string"}

#@markdown ## AlphaFold Parameters
num_recycles = 3  #@param {type:"raw"}
use_multimer = True  #@param {type:"boolean"}
initial_guess = True  #@param {type:"boolean"}
use_template = False  #@param {type:"boolean"}

#@markdown ## Google Drive Settings
use_google_drive = True  #@param {type:"boolean"}
drive_output_folder = "RFDiffusion_Results"  #@param {type:"string"}

#@markdown ## Incremental Save Settings
save_incrementally = True  #@param {type:"boolean"}

# =============================================================================
# DERIVED VALUES
# =============================================================================
import os
import shutil
from string import ascii_uppercase, ascii_lowercase
from datetime import datetime

design_positions_structure = [r - receptor_start_pdb for r in design_residues_pdb]
RECEPTOR_LEN = len(structure_receptor)
EFFECTOR_LEN = len(structure_effector)
native_lrr = structure_receptor
native_effector = structure_effector
insertion_positions = design_positions_structure
chains = f"{receptor_chain},{effector_chain}"
alphabet_list = list(ascii_uppercase + ascii_lowercase)
custom_template_path = f"{pdb_id.lower()}.pdb"
run_folder = None

print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"\nProject: {project_name}")
print(f"PDB: {pdb_id}")
print(f"Receptor: {RECEPTOR_LEN} residues, Effector: {EFFECTOR_LEN} residues")
print(f"Design positions (PDB): {design_residues_pdb}")
print(f"Design positions (0-indexed): {design_positions_structure}")
print(f"RFDiffusion: {num_designs} designs, {iterations} iterations")
print(f"MPNN: {num_seqs} sequences per design")
print(f"AlphaFold: {num_recycles} recycles, template={use_template}")
print(f"\n** INCREMENTAL SAVING: {'ENABLED' if save_incrementally else 'DISABLED'} **")
print("=" * 60)

CONFIGURATION SUMMARY

Project: Sr35_AvrSr35_interface_recapitulation
PDB: 7XC2
Receptor: 355 residues, Effector: 431 residues
Design positions (PDB): [654, 673, 730, 755, 803, 809, 854, 856, 858]
Design positions (0-indexed): [92, 111, 168, 193, 241, 247, 292, 294, 296]
RFDiffusion: 32 designs, 100 iterations
MPNN: 2 sequences per design
AlphaFold: 3 recycles, template=True

** INCREMENTAL SAVING: ENABLED **


In [ ]:
#@title Cell 1: Upload PDB File & Setup Run Folder
import shutil
import os
from datetime import datetime

if os.path.exists(custom_template_path):
    os.remove(custom_template_path)

if use_template:
    print(f"Template mode ENABLED")
else:
    print(f"Template mode DISABLED")

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_path = f"/content/drive/MyDrive/{drive_output_folder}"
    os.makedirs(drive_path, exist_ok=True)

    # CREATE RUN FOLDER NOW (at the very start)
    if save_incrementally:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_folder = f"{drive_path}/{project_name}_{timestamp}"
        os.makedirs(run_folder, exist_ok=True)
        os.makedirs(f"{run_folder}/rfdiffusion_backbones", exist_ok=True)
        os.makedirs(f"{run_folder}/mpnn_sequences", exist_ok=True)
        os.makedirs(f"{run_folder}/alphafold_results", exist_ok=True)
        os.makedirs(f"{run_folder}/alphafold_results/all_pdb", exist_ok=True)
        print(f"\n{'='*60}")
        print(f"✓ INCREMENTAL SAVE FOLDER CREATED")
        print(f"  {run_folder}")
        print(f"{'='*60}")

    print(f"✓ Google Drive mounted")

    drive_pdb_path = f"{drive_path}/{pdb_id.lower()}.pdb"
    alt_drive_pdb_path = f"{drive_path}/{pdb_id.upper()}.pdb"

    if os.path.exists(drive_pdb_path):
        shutil.copy(drive_pdb_path, custom_template_path)
        print(f"✓ Found PDB from Drive: {drive_pdb_path}")
    elif os.path.exists(alt_drive_pdb_path):
        shutil.copy(alt_drive_pdb_path, custom_template_path)
        print(f"✓ Found PDB from Drive: {alt_drive_pdb_path}")
    else:
        print(f"\nPDB not found in Drive folder, falling back to browser upload...")
        from google.colab import files
        uploaded = files.upload()
        filename = next(iter(uploaded))
        shutil.move(filename, custom_template_path)
        shutil.copy(custom_template_path, drive_pdb_path)
        print(f"✓ PDB saved to Drive for future runs")

    if run_folder:
        shutil.copy(custom_template_path, f"{run_folder}/input_{pdb_id}.pdb")
        config_text = f"""# RFDiffusion Pipeline Configuration
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
project_name = "{project_name}"
pdb_id = "{pdb_id}"
receptor_chain = "{receptor_chain}"
effector_chain = "{effector_chain}"
design_residues_pdb = {design_residues_pdb}
receptor_start_pdb = {receptor_start_pdb}
contigs = "{contigs}"
hotspot = "{hotspot}"
num_designs = {num_designs}
iterations = {iterations}
num_seqs = {num_seqs}
mpnn_sampling_temp = {mpnn_sampling_temp}
rm_aa = "{rm_aa}"
num_recycles = {num_recycles}
use_multimer = {use_multimer}
use_template = {use_template}
"""
        with open(f"{run_folder}/config.txt", "w") as f:
            f.write(config_text)
        print(f"✓ Configuration saved")
else:
    from google.colab import files
    print(f"\nUpload your PDB file for {pdb_id}...")
    uploaded = files.upload()
    filename = next(iter(uploaded))
    shutil.move(filename, custom_template_path)

print(f"✓ PDB prepared as: {os.path.abspath(custom_template_path)}")


Template mode ENABLED
Mounted at /content/drive

✓ INCREMENTAL SAVE FOLDER CREATED
  /content/drive/MyDrive/RFDiffusion_Results/Sr35_AvrSr35_interface_recapitulation_20260127_113005
✓ Google Drive mounted
✓ Found PDB from Drive: /content/drive/MyDrive/RFDiffusion_Results/7xc2.pdb
✓ Configuration saved
✓ PDB prepared as: /content/7xc2.pdb


In [ ]:
#@title Cell 2: Setup RFDiffusion
import os, time, signal
import sys, random, string, re

if not os.path.isdir("params"):
    os.system("apt-get install aria2")
    os.system("mkdir params")
    os.system("(aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; tar -xf alphafold_params_2022-12-06.tar -C params; touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
    print("installing RFdiffusion...")
    os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
    os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
    os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
    os.system("wget -qnc https://files.ipd.uw.edu/krypton/ananas")
    os.system("chmod +x ananas")

if not os.path.isdir("colabdesign"):
    print("installing ColabDesign...")
    os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
    print("downloading RFdiffusion params...")
    os.system("mkdir RFdiffusion/models")
    models = ["Base_ckpt.pt","Complex_base_ckpt.pt"]
    for m in models:
        while os.path.isfile(f"{m}.aria2"):
            time.sleep(5)
    os.system(f"mv {' '.join(models)} RFdiffusion/models")
    os.system("unzip schedules.zip; rm schedules.zip")

if 'RFdiffusion' not in sys.path:
    os.environ["DGLBACKEND"] = "pytorch"
    sys.path.append('RFdiffusion')

from google.colab import files
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol

from inference.utils import parse_pdb
from colabdesign.rf.utils import get_ca
from colabdesign.rf.utils import fix_contigs, fix_partial_contigs, fix_pdb, sym_it
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

def get_pdb(pdb_code=None):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    if not os.path.isfile(f"{pdb_code}.pdb1"):
      os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
      os.system(f"gunzip {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"
  else:
    os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v3.pdb")
    return f"AF-{pdb_code}-F1-model_v3.pdb"

def run_ananas(pdb_str, path, sym=None):
  pdb_filename = f"outputs/{path}/ananas_input.pdb"
  out_filename = f"outputs/{path}/ananas.json"
  with open(pdb_filename,"w") as handle:
    handle.write(pdb_str)
  cmd = f"./ananas {pdb_filename} -u -j {out_filename}"
  if sym is None: os.system(cmd)
  else: os.system(f"{cmd} {sym}")
  try:
    out = json.loads(open(out_filename,"r").read())
    results,AU = out[0], out[-1]["AU"]
    group = AU["group"]
    chains = AU["chain names"]
    rmsd = results["Average_RMSD"]
    print(f"AnAnaS detected {group} symmetry at RMSD:{rmsd:.3}")
    C = np.array(results['transforms'][0]['CENTER'])
    A = [np.array(t["AXIS"]) for t in results['transforms']]
    new_lines = []
    for line in pdb_str.split("\n"):
      if line.startswith("ATOM"):
        chain = line[21:22]
        if chain in chains:
          x = np.array([float(line[i:(i+8)]) for i in [30,38,46]])
          if group[0] == "c": x = sym_it(x,C,A[0])
          if group[0] == "d": x = sym_it(x,C,A[1],A[0])
          coord_str = "".join(["{:8.3f}".format(a) for a in x])
          new_lines.append(line[:30]+coord_str+line[54:])
      else:
        new_lines.append(line)
    return results, "\n".join(new_lines)
  except:
    return None, pdb_str

def run(command, steps, num_designs=1, visual="none"):
  def run_command_and_get_pid(command):
    pid_file = '/dev/shm/pid'
    os.system(f'nohup {command} > /dev/null & echo $! > {pid_file}')
    with open(pid_file, 'r') as f: pid = int(f.read().strip())
    os.remove(pid_file)
    return pid
  def is_process_running(pid):
    try: os.kill(pid, 0)
    except OSError: return False
    else: return True

  run_output = widgets.Output()
  progress = widgets.FloatProgress(min=0, max=1, description='running', bar_style='info')
  display(widgets.VBox([progress, run_output]))

  for n in range(steps):
    if os.path.isfile(f"/dev/shm/{n}.pdb"): os.remove(f"/dev/shm/{n}.pdb")

  pid = run_command_and_get_pid(command)
  try:
    fail = False
    for _ in range(num_designs):
      for n in range(steps):
        wait = True
        while wait and not fail:
          time.sleep(0.1)
          if os.path.isfile(f"/dev/shm/{n}.pdb"):
            pdb_str = open(f"/dev/shm/{n}.pdb").read()
            if pdb_str[-3:] == "TER": wait = False
            elif not is_process_running(pid): fail = True
          elif not is_process_running(pid): fail = True
        if fail:
          progress.bar_style = 'danger'
          progress.description = "failed"
          break
        else:
          progress.value = (n+1) / steps
          if visual != "none":
            with run_output:
              run_output.clear_output(wait=True)
              if visual == "image":
                xyz, bfact = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                fig = plt.figure()
                fig.set_dpi(100);fig.set_figwidth(6);fig.set_figheight(6)
                ax1 = fig.add_subplot(111);ax1.set_xticks([]);ax1.set_yticks([])
                plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                plt.show()
              if visual == "interactive":
                view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
                view.addModel(pdb_str,'pdb')
                view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
                view.zoomTo()
                view.show()
        if os.path.exists(f"/dev/shm/{n}.pdb"): os.remove(f"/dev/shm/{n}.pdb")
      if fail:
        progress.bar_style = 'danger'
        progress.description = "failed"
        break
    while is_process_running(pid): time.sleep(0.1)
  except KeyboardInterrupt:
    os.kill(pid, signal.SIGTERM)
    progress.bar_style = 'danger'
    progress.description = "stopped"

def run_diffusion(contigs, path, pdb=None, iterations=50, symmetry="none", order=1, hotspot=None, chains=None, add_potential=False, num_designs=1, visual="none"):
  full_path = f"outputs/{path}"
  os.makedirs(full_path, exist_ok=True)
  opts = [f"inference.output_prefix={full_path}", f"inference.num_designs={num_designs}"]
  if chains == "": chains = None

  if symmetry in ["auto","cyclic","dihedral"]:
    if symmetry == "auto": sym, copies = None, 1
    else: sym, copies = {"cyclic":(f"c{order}",order), "dihedral":(f"d{order}",order*2)}[symmetry]
  else:
    symmetry = None
    sym, copies = None, 1

  contigs = contigs.replace(","," ").replace(":"," ").split()
  is_fixed, is_free = False, False
  fixed_chains = []
  for contig in contigs:
    for x in contig.split("/"):
      a = x.split("-")[0]
      if a[0].isalpha():
        is_fixed = True
        if a[0] not in fixed_chains: fixed_chains.append(a[0])
      if a.isnumeric(): is_free = True
  if len(contigs) == 0 or not is_free: mode = "partial"
  elif is_fixed: mode = "fixed"
  else: mode = "free"

  if mode in ["partial","fixed"]:
    pdb_str = pdb_to_string(get_pdb(pdb), chains=chains)
    if symmetry == "auto":
      a, pdb_str = run_ananas(pdb_str, path)
      if a is None:
        print(f'ERROR: no symmetry detected')
        symmetry = None
        sym, copies = None, 1
      else:
        if a["group"][0] == "c":
          symmetry = "cyclic"
          sym, copies = a["group"], int(a["group"][1:])
        elif a["group"][0] == "d":
          symmetry = "dihedral"
          sym, copies = a["group"], 2 * int(a["group"][1:])
        else:
          print(f'ERROR: the detected symmetry ({a["group"]}) not currently supported')
          symmetry = None
          sym, copies = None, 1
    elif mode == "fixed":
      pdb_str = pdb_to_string(pdb_str, chains=fixed_chains)

    pdb_filename = f"{full_path}/input.pdb"
    with open(pdb_filename, "w") as handle: handle.write(pdb_str)

    parsed_pdb = parse_pdb(pdb_filename)
    opts.append(f"inference.input_pdb={pdb_filename}")
    if mode in ["partial"]:
      iterations = int(80 * (iterations / 200))
      opts.append(f"diffuser.partial_T={iterations}")
      contigs = fix_partial_contigs(contigs, parsed_pdb)
    else:
      opts.append(f"diffuser.T={iterations}")
      contigs = fix_contigs(contigs, parsed_pdb)
  else:
    opts.append(f"diffuser.T={iterations}")
    parsed_pdb = None
    contigs = fix_contigs(contigs, parsed_pdb)

  if hotspot is not None and hotspot != "": opts.append(f"ppi.hotspot_res=[{hotspot}]")

  if sym is not None:
    sym_opts = ["--config-name symmetry", f"inference.symmetry={sym}"]
    if add_potential:
      sym_opts += ["'potentials.guiding_potentials=[\"type:olig_contacts,weight_intra:1,weight_inter:0.1\"]'",
                   "potentials.olig_intra_all=True","potentials.olig_inter_all=True",
                   "potentials.guide_scale=2","potentials.guide_decay=quadratic"]
    opts = sym_opts + opts
    contigs = sum([contigs] * copies,[])

  opts.append(f"'contigmap.contigs=[{' '.join(contigs)}]'")
  opts += ["inference.dump_pdb=True","inference.dump_pdb_path='/dev/shm'"]

  print("mode:", mode)
  print("output:", full_path)
  print("contigs:", contigs)

  opts_str = " ".join(opts)
  cmd = f"./RFdiffusion/run_inference.py {opts_str}"
  print(cmd)
  run(cmd, iterations, num_designs, visual=visual)

  for n in range(num_designs):
    pdbs = [f"outputs/traj/{path}_{n}_pX0_traj.pdb", f"outputs/traj/{path}_{n}_Xt-1_traj.pdb", f"{full_path}_{n}.pdb"]
    for pdb in pdbs:
      with open(pdb,"r") as handle: pdb_str = handle.read()
      with open(pdb,"w") as handle: handle.write(fix_pdb(pdb_str, contigs))

  return contigs, copies

installing RFdiffusion...
installing ColabDesign...
downloading RFdiffusion params...


In [ ]:
#@title Cell 3: Run RFDiffusion
name = project_name
pdb = pdb_id
path = name
while os.path.exists(f"outputs/{path}_0.pdb"):
    path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

flags = {"contigs": contigs, "pdb": pdb, "order": order, "iterations": iterations, "symmetry": symmetry,
         "hotspot": hotspot, "path": path, "chains": chains, "add_potential": add_potential,
         "num_designs": num_designs, "visual": visual}

for k, v in flags.items():
    if isinstance(v, str): flags[k] = v.replace("'", "").replace('"', '')

contigs, copies = run_diffusion(**flags)

print(f"\n✓ RFDiffusion complete!")
print(f"  Output path: outputs/{path}")

# INCREMENTAL SAVE: RFDiffusion backbone PDBs
if run_folder and save_incrementally:
    print(f"\n📁 Saving RFDiffusion backbones to Google Drive...")
    backbone_folder = f"{run_folder}/rfdiffusion_backbones"
    for n in range(num_designs):
        src_pdb = f"outputs/{path}_{n}.pdb"
        if os.path.exists(src_pdb):
            shutil.copy(src_pdb, f"{backbone_folder}/design_{n}.pdb")
    input_pdb = f"outputs/{path}/input.pdb"
    if os.path.exists(input_pdb):
        shutil.copy(input_pdb, f"{backbone_folder}/rfdiffusion_input.pdb")
    print(f"✓ Saved {num_designs} backbone PDBs to: {backbone_folder}")

mode: fixed
output: outputs/Sr35_AvrSr35_interface_recapitulation
contigs: ['B128-246/B250-561', 'A562-653/1-1/A655-672/1-1/A674-729/1-1/A731-754/1-1/A756-802/1-1/A804-808/1-1/A810-853/1-1/A855-855/1-1/A857-857/1-1/A859-865/A869-919']
./RFdiffusion/run_inference.py inference.output_prefix=outputs/Sr35_AvrSr35_interface_recapitulation inference.num_designs=32 inference.input_pdb=outputs/Sr35_AvrSr35_interface_recapitulation/input.pdb diffuser.T=100 ppi.hotspot_res=[B292,B293,B294,B295,B296,B297,B344,B345,B346,B347,B348,B379,B380,B381,B382,B383,B384,B391,B392,B393,B394,B395,B396] 'contigmap.contigs=[B128-246/B250-561 A562-653/1-1/A655-672/1-1/A674-729/1-1/A731-754/1-1/A756-802/1-1/A804-808/1-1/A810-853/1-1/A855-855/1-1/A857-857/1-1/A859-865/A869-919]' inference.dump_pdb=True inference.dump_pdb_path='/dev/shm'


In [ ]:
#@title Cell 4: Display 3D Structure (Optional)
animate = "interactive"  #@param ["none", "movie", "interactive"]
color = "chain"  #@param ["rainbow", "chain", "plddt"]
denoise = True
dpi = 100  #@param {type:"raw"}
from colabdesign.shared.plot import pymol_color_list
from colabdesign.rf.utils import get_ca, get_Ls, make_animation

def plot_pdb(num=0):
  if denoise: pdb_traj = f"outputs/traj/{path}_{num}_pX0_traj.pdb"
  else: pdb_traj = f"outputs/traj/{path}_{num}_Xt-1_traj.pdb"
  if animate in ["none","interactive"]:
    hbondCutoff = 4.0
    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    if animate == "interactive":
      pdb_str = open(pdb_traj,'r').read()
      view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    else:
      pdb = f"outputs/{path}_{num}.pdb"
      pdb_str = open(pdb,'r').read()
      view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    if color == "rainbow": view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
      for n,chain,c in zip(range(len(contigs)), alphabet_list, pymol_color_list):
          view.setStyle({'chain':chain},{'cartoon': {'color':c}})
    else: view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
    view.zoomTo()
    if animate == "interactive": view.animate({'loop': 'backAndForth'})
    view.show()
  else:
    Ls = get_Ls(contigs)
    xyz, bfact = get_ca(pdb_traj, get_bfact=True)
    xyz = xyz.reshape((-1,sum(Ls),3))[::-1]
    bfact = bfact.reshape((-1,sum(Ls)))[::-1]
    if color == "chain": display(HTML(make_animation(xyz, Ls=Ls, dpi=dpi, ref=-1)))
    elif color == "rainbow": display(HTML(make_animation(xyz, dpi=dpi, ref=-1)))
    else: display(HTML(make_animation(xyz, plddt=bfact*100, dpi=dpi, ref=-1)))

if num_designs > 1:
  output = widgets.Output()
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(options=[(f'{k}',k) for k in range(num_designs)], value=0, description='design:')
  dropdown.observe(on_change)
  display(widgets.VBox([dropdown, output]))
  with output: plot_pdb(dropdown.value)
else:
  plot_pdb()


In [ ]:
#@title Cell 5: Setup Sequence Analysis Functions
import numpy as np
import pandas as pd
from Bio import Align
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af import mk_af_model
from colabdesign.shared.protein import pdb_to_string

def align_sequences(seq1, seq2):
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -0.5
    alignments = aligner.align(seq1, seq2)
    best = alignments[0]
    return str(best[0]), str(best[1])

def analyze_mpnn_changes_aligned(mpnn_lrr, native_lrr, designed_positions):
    aligned_native, aligned_mpnn = align_sequences(native_lrr, mpnn_lrr)
    intended_changes = []
    unintended_changes = []
    native_idx = 0
    for aligned_native_char, aligned_mpnn_char in zip(aligned_native, aligned_mpnn):
        if aligned_native_char == '-': continue
        if aligned_mpnn_char == '-':
            change = {'position': native_idx, 'native': aligned_native_char, 'mpnn': '-', 'type': 'deletion'}
            if native_idx in designed_positions: intended_changes.append(change)
            else: unintended_changes.append(change)
        elif aligned_native_char != aligned_mpnn_char:
            change = {'position': native_idx, 'native': aligned_native_char, 'mpnn': aligned_mpnn_char, 'type': 'substitution'}
            if native_idx in designed_positions: intended_changes.append(change)
            else: unintended_changes.append(change)
        native_idx += 1
    return {'intended': intended_changes, 'unintended': unintended_changes, 'mpnn_length': len(mpnn_lrr),
            'native_length': len(native_lrr), 'aligned_native': aligned_native, 'aligned_mpnn': aligned_mpnn}

def restore_native_except_designed_aligned(mpnn_lrr, native_lrr, designed_positions):
    aligned_native, aligned_mpnn = align_sequences(native_lrr, mpnn_lrr)
    designed_set = set(designed_positions)
    corrected = []
    native_idx = 0
    for aligned_native_char, aligned_mpnn_char in zip(aligned_native, aligned_mpnn):
        if aligned_native_char == '-': continue
        if native_idx in designed_set:
            if aligned_mpnn_char == '-':
                print(f"  WARNING: MPNN deleted designed position {native_idx}, using native")
                corrected.append(aligned_native_char)
            else: corrected.append(aligned_mpnn_char)
        else: corrected.append(aligned_native_char)
        native_idx += 1
    return ''.join(corrected)

def process_mpnn_output(mpnn_seq, native_effector, native_lrr, designed_positions):
    if '/' in mpnn_seq: parts = mpnn_seq.split('/')
    elif ':' in mpnn_seq: parts = mpnn_seq.split(':')
    else:
        print("ERROR: Could not find chain separator")
        return None, None, None, False
    if len(parts) < 2:
        print(f"ERROR: Expected 2 chains, got {len(parts)}")
        return None, None, None, False
    effector_pred = parts[0]
    lrr_pred = parts[1]
    print(f"    Effector length: {len(effector_pred)} (native: {len(native_effector)})")
    print(f"    LRR length: {len(lrr_pred)} (native: {len(native_lrr)})")
    analysis = analyze_mpnn_changes_aligned(lrr_pred, native_lrr, designed_positions)
    print(f"\n    --- MPNN Change Analysis ---")
    print(f"    Intended changes: {len(analysis['intended'])}")
    print(f"    Unintended changes: {len(analysis['unintended'])}")
    if analysis['unintended']:
        for c in analysis['unintended'][:5]:
            print(f"      Position {c['position']}: {c['native']} -> {c['mpnn']} ({c['type']})")
        if len(analysis['unintended']) > 5: print(f"      ... and {len(analysis['unintended']) - 5} more")
    corrected_lrr = restore_native_except_designed_aligned(lrr_pred, native_lrr, designed_positions)
    return effector_pred, corrected_lrr, analysis, True

print("✓ Sequence analysis functions loaded!")
print(f"  Design positions (0-indexed): {insertion_positions}")
print(f"  Native residues at positions: {''.join([native_lrr[i] for i in insertion_positions])}")

In [ ]:
#@title Cell 6: Run ProteinMPNN

def get_info(contig):
    F = []
    free_chain = False
    fixed_chain = False
    for segment in contig.split("/"):
        if not segment or segment == "0": continue
        if segment[0].isalpha():
            chain_letter = segment[0]
            residue_part = segment[1:]
            if "-" in residue_part:
                parts = residue_part.split("-")
                start = int(parts[0])
                end_str = parts[1]
                if end_str == "": L = 1
                else: L = int(end_str) - start + 1
            else: L = 1
            F += [1] * L
            fixed_chain = True
        else:
            if "-" in segment: L = int(segment.split("-")[1])
            else: L = int(segment)
            F += [0] * L
            free_chain = True
    return F, [fixed_chain, free_chain]

if not os.path.isfile("params/done.txt"):
    print("Waiting for AlphaFold params...")
    while not os.path.isfile("params/done.txt"): time.sleep(5)

pdb_input = f"outputs/{path}_0.pdb"
loc_dir = f"outputs/{path}"
num_designs_local = num_designs
copies_local = copies

parsed_contigs = contigs if isinstance(contigs, list) else contigs.split()
chains_list = alphabet_list[:len(parsed_contigs)]
print(f"Parsed {len(parsed_contigs)} chain(s): {chains_list}")

info = [get_info(x) for x in parsed_contigs]
fixed_pos = []
for pos, (fixed_chain, free_chain) in info: fixed_pos += pos
print(f"Design positions: {fixed_pos.count(0)}, Fixed positions: {fixed_pos.count(1)}")

flags = {"initial_guess": initial_guess, "best_metric": "rmsd", "use_multimer": use_multimer,
         "model_names": ["model_1_multimer_v3" if use_multimer else "model_1_ptm"]}

af_model = mk_af_model(protocol="fixbb", **flags)
prep_flags = {"chain": ",".join(chains_list), "copies": copies_local, "homooligomer": copies_local > 1,
              "rm_aa": rm_aa if rm_aa else None}

batch_size = min(8, num_seqs)

print(f"\nRunning ProteinMPNN...")
mpnn_model = mk_mpnn_model(model_name="v_48_020")
outs = []
pdbs = []

for m in range(num_designs_local):
    pdb_filename = pdb_input.replace("_0.pdb", f"_{m}.pdb") if num_designs_local > 0 else pdb_input
    pdbs.append(pdb_filename)
    print(f"  Processing design {m}: {pdb_filename}")
    af_model.prep_inputs(pdb_filename, **prep_flags)
    mpnn_model.get_af_inputs(af_model)
    S = mpnn_model._inputs['S']
    n_pos = len(S)
    n_aa = 20
    design_positions_full = [EFFECTOR_LEN + pos for pos in insertion_positions]
    print(f"    Design positions in full seq: {design_positions_full}")
    bias = np.zeros((n_pos, n_aa))
    fix_bias = -1000.0
    for i in range(n_pos):
        native_aa = S[i]
        if i not in design_positions_full:
            bias[i, :] = fix_bias
            bias[i, native_aa] = 0.0
    mpnn_model._inputs['bias'] = bias
    outs.append(mpnn_model.sample(num=num_seqs // batch_size, batch=batch_size, temperature=mpnn_sampling_temp))

total_sequences = sum(len(out["seq"]) for out in outs)
print(f"\n✓ MPNN complete! Generated {total_sequences} total sequences.")

# =============================================================================
# INCREMENTAL SAVE: ProteinMPNN Sequences (BEFORE AlphaFold)
# =============================================================================
if run_folder and save_incrementally:
    print(f"\n📁 Saving ProteinMPNN sequences to Google Drive...")
    mpnn_folder = f"{run_folder}/mpnn_sequences"

    raw_fasta_lines = []
    corrected_fasta_lines = []
    sequence_metadata = []
    native_at_design = [native_lrr[i] for i in insertion_positions]

    for m, out in enumerate(outs):
        for n in range(num_seqs):
            mpnn_seq = out["seq"][n]
            mpnn_score = out["score"][n]

            raw_fasta_lines.append(f">design_{m}_seq_{n}|mpnn_score={mpnn_score:.3f}|raw")
            raw_fasta_lines.append(mpnn_seq)

            effector_pred, corrected_lrr, analysis, success = process_mpnn_output(
                mpnn_seq, native_effector, native_lrr, design_positions_structure
            )

            if success:
                corrected_full_seq = f"{corrected_lrr}:{effector_pred}"
                intended_dict = {c['position']: c for c in analysis['intended']}
                designed_residues = []
                for pos in insertion_positions:
                    if pos in intended_dict: designed_residues.append(intended_dict[pos]['mpnn'])
                    else: designed_residues.append(native_lrr[pos])

                num_changes = sum(1 for d, na in zip(designed_residues, native_at_design) if d != na)
                designed_str = ''.join(designed_residues)
                native_str = ''.join(native_at_design)

                corrected_fasta_lines.append(f">design_{m}_seq_{n}|mpnn_score={mpnn_score:.3f}|designed={designed_str}|native={native_str}|changes={num_changes}")
                corrected_fasta_lines.append(corrected_full_seq)

                sequence_metadata.append({
                    'design': m, 'seq': n, 'mpnn_score': mpnn_score, 'native_residues': native_str,
                    'designed_residues': designed_str, 'num_changes': num_changes, 'corrected_sequence': corrected_full_seq
                })
            else:
                corrected_fasta_lines.append(f">design_{m}_seq_{n}|mpnn_score={mpnn_score:.3f}|ERROR")
                corrected_fasta_lines.append(mpnn_seq)

    with open(f"{mpnn_folder}/mpnn_raw.fasta", "w") as f:
        f.write("\n".join(raw_fasta_lines))

    with open(f"{mpnn_folder}/mpnn_corrected_for_alphafold.fasta", "w") as f:
        f.write("\n".join(corrected_fasta_lines))

    if sequence_metadata:
        meta_df = pd.DataFrame(sequence_metadata)
        meta_df.to_csv(f"{mpnn_folder}/sequence_metadata.csv", index=False)

    with open(f"{mpnn_folder}/colabfold_batch.csv", "w") as f:
        f.write("id,sequence\n")
        for meta in sequence_metadata:
            seq_id = f"design_{meta['design']}_seq_{meta['seq']}"
            f.write(f"{seq_id},{meta['corrected_sequence']}\n")

    print(f"✓ Saved to: {mpnn_folder}")
    print(f"  - mpnn_raw.fasta ({total_sequences} raw sequences)")
    print(f"  - mpnn_corrected_for_alphafold.fasta ({len(sequence_metadata)} corrected)")
    print(f"  - sequence_metadata.csv")
    print(f"  - colabfold_batch.csv (⭐ Ready for HPC AlphaFold!)")

In [ ]:
#@title Cell 7: AlphaFold Validation with Incremental Saving
import warnings
import glob
import hashlib
warnings.filterwarnings("ignore", message="Explicitly requested dtype.*int64.*")
warnings.simplefilter(action='ignore', category=FutureWarning)

os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")

if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold 2>/dev/null || true")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold 2>/dev/null || true")
    os.system("touch COLABFOLD_READY")

if not os.path.isfile("CONDA_READY"):
    print("Installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if not os.path.isfile("HHSUITE_READY"):
    print("Installing hhsuite...")
    os.system("mamba install -y -c conda-forge -c bioconda hhsuite=3.3.0")
    os.system("touch HHSUITE_READY")

from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.batch import get_queries, run, set_model_type
from pathlib import Path

print("Running AlphaFold validation...")
if use_template: print(f"  Template: {custom_template_path}")
else: print(f"  Template: DISABLED")

def save_incremental_af_results(data, labels, best, loc_dir, run_folder, fasta_lines=None):
    if not run_folder: return
    try:
        af_folder = f"{run_folder}/alphafold_results"
        df = pd.DataFrame(data, columns=labels)
        df = df.sort_values(by="ranking_score", ascending=False)
        df.to_csv(f'{loc_dir}/design_validation_results.csv', index=False)
        shutil.copy(f'{loc_dir}/design_validation_results.csv', f'{af_folder}/design_validation_results.csv')
        if best.get("score", -1) > 0:
            best_pdb_local = f"{loc_dir}/best.pdb"
            if os.path.exists(best_pdb_local): shutil.copy(best_pdb_local, f"{af_folder}/best.pdb")
        if fasta_lines:
            with open(f"{af_folder}/validated.fasta", "w") as f: f.write("\n".join(fasta_lines))
        print(f"    ✓ Incremental save ({len(data)} results)")
    except Exception as e:
        print(f"    ⚠ Save failed: {e}")

labels = ["design", "n", "score", "ranking_score", "plddt", "ptm", "iptm", "pae",
          "hash", "native_residues", "designed_residues", "num_changes", "pct_change", "designed_receptor"]

data = []
best = {"ptm": -1, "iptm": -1, "score": -1, "design": 0, "n": 0}
best_pdb_path = None
fasta_lines = []

os.system(f"mkdir -p {loc_dir}/all_pdb")
native_at_design = [native_lrr[i] for i in insertion_positions]
output_log_path = f"{loc_dir}/sequence_correction_log.txt"

with open(f"{loc_dir}/design.fasta", "w") as fasta, open(output_log_path, 'w') as log_file:
    def log(msg=""):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("=" * 80)
    log(f"ALPHAFOLD VALIDATION - {project_name}")
    log(f"Template: {'ENABLED' if use_template else 'DISABLED'}")
    log(f"Incremental Saving: {'ENABLED' if run_folder else 'DISABLED'}")
    log("=" * 80)

    for m, out in enumerate(outs):
        out["design"] = []
        out["n"] = []
        out["plddt"] = []
        out["ptm"] = []
        out["iptm"] = []
        out["pae"] = []
        out["corrected_seq"] = []
        out["analysis"] = []

        log(f"\n{'='*80}")
        log(f"--- Design {m} ---")
        log(f"{'='*80}")

        for n in range(num_seqs):
            log(f"\n  Sequence {n}:")
            out["design"].append(m)
            out["n"].append(n)

            mpnn_seq = out["seq"][n]
            effector_pred, corrected_lrr, analysis, success = process_mpnn_output(
                mpnn_seq, native_effector, native_lrr, design_positions_structure
            )

            if success:
                corrected_full_seq = f"{corrected_lrr}:{effector_pred}"
                out["corrected_seq"].append(corrected_full_seq)
                out["analysis"].append(analysis)
            else:
                corrected_full_seq = mpnn_seq
                out["corrected_seq"].append(mpnn_seq)
                out["analysis"].append(None)

            if success and analysis:
                intended_dict = {c['position']: c for c in analysis['intended']}
                designed_residues = []
                for pos in insertion_positions:
                    if pos in intended_dict: designed_residues.append(intended_dict[pos]['mpnn'])
                    else: designed_residues.append(native_lrr[pos])
                num_changes = sum(1 for d, na in zip(designed_residues, native_at_design) if d != na)
                log(f"    Designed: {''.join(designed_residues)}, Changes: {num_changes}/{len(insertion_positions)}")

            seq_hash = hashlib.md5(corrected_full_seq.encode()).hexdigest()[:8]

            log(f"    --- AlphaFold Prediction ---")
            temp_query_file = f"{loc_dir}/temp_query_{m}_{n}.csv"
            with open(temp_query_file, "w") as qf:
                qf.write(f"id,sequence\ndesign_{m}_n_{n},{corrected_full_seq}")

            try:
                queries, is_complex = get_queries(temp_query_file)
                model_type = set_model_type(is_complex, "alphafold2_multimer_v3")
                download_alphafold_params(model_type, Path("."))

                template_dir = None
                if use_template and custom_template_path and os.path.isfile(custom_template_path):
                    template_dir = f"{loc_dir}/templates_{m}_{n}"
                    os.makedirs(template_dir, exist_ok=True)
                    os.system(f"cp {custom_template_path} {template_dir}/")

                current_result_dir = f"{loc_dir}/colabfold_{m}_{n}"

                results = run(queries=queries, result_dir=current_result_dir,
                    use_templates=use_template and (template_dir is not None),
                    custom_template_path=template_dir if use_template else None,
                    num_relax=0, msa_mode="single_sequence", model_type=model_type,
                    num_models=5, num_recycles=num_recycles, relax_max_iterations=0,
                    num_seeds=1, use_dropout=False, is_complex=is_complex,
                    data_dir=Path("."), keep_existing_results=False, rank_by="auto",
                    pair_mode="unpaired_paired", pairing_strategy="greedy",
                    stop_at_score=float(100), dpi=150, zip_results=False,
                    save_all=False, use_cluster_profile=True, save_recycles=False,
                    user_agent="colabfold/rfdiffusion-validation")

                pdb_files = sorted(glob.glob(f"{current_result_dir}/*_rank_001*.pdb"))
                if not pdb_files: pdb_files = sorted(glob.glob(f"{current_result_dir}/*.pdb"))
                best_pdb = pdb_files[0] if pdb_files else None
                json_files = sorted(glob.glob(f"{current_result_dir}/*_rank_001*.json"))

                plddt_avg, ptm, iptm, pae = 0.0, 0.0, 0.0, 31.0
                if json_files:
                    with open(json_files[0], 'r') as f:
                        scores = json.load(f)
                        plddt_avg = np.mean(scores.get("plddt", [0]))
                        ptm = scores.get("ptm", 0)
                        iptm = scores.get("iptm", 0)
                        pae = np.mean(scores.get("pae", [31]))

                ranking_score = (0.8 * iptm) + (0.2 * ptm)
                out["plddt"].append(plddt_avg)
                out["ptm"].append(ptm)
                out["iptm"].append(iptm)
                out["pae"].append(pae)

                log(f"    pLDDT: {plddt_avg:.3f}, pTM: {ptm:.3f}, iPTM: {iptm:.3f}, Ranking: {ranking_score:.3f}")

                if ranking_score > best.get("score", -1):
                    best = {"design": m, "n": n, "ptm": ptm, "iptm": iptm, "plddt": plddt_avg, "score": ranking_score}
                    if best_pdb:
                        with open(f"{loc_dir}/best.pdb", "w") as handle:
                            handle.write(f"REMARK 001 design {m} N {n} pTM {ptm:.3f} iPTM {iptm:.3f} pLDDT {plddt_avg:.3f}\n")
                            handle.write(open(best_pdb).read())

                if best_pdb:
                    os.system(f"cp {best_pdb} {loc_dir}/all_pdb/design{m}_n{n}.pdb")
                    if run_folder: shutil.copy(best_pdb, f"{run_folder}/alphafold_results/all_pdb/design{m}_n{n}.pdb")

                score_line = f'>design:{m}|n:{n}|mpnn:{out["score"][n]:.3f}|plddt:{plddt_avg:.3f}|ptm:{ptm:.3f}|iptm:{iptm:.3f}'
                fasta.write(f"{score_line}\n{corrected_full_seq}\n")
                fasta.flush()
                fasta_lines.append(f"{score_line}\n{corrected_full_seq}")

                os.remove(temp_query_file)

                native_str = ''.join(native_at_design)
                designed_str = ''.join(designed_residues) if success else "N/A"
                n_changes = num_changes if success else 0
                p_change = (n_changes / len(insertion_positions)) * 100 if len(insertion_positions) > 0 else 0

                row_data = [m, n, out["score"][n], ranking_score, plddt_avg, ptm, iptm, pae, seq_hash,
                           native_str, designed_str, n_changes, f"{p_change:.1f}%",
                           corrected_lrr if success else "Error"]
                data.append(row_data)

                if save_incrementally and run_folder:
                    save_incremental_af_results(data, labels, best, loc_dir, run_folder, fasta_lines)
                    log_file.flush()
                    shutil.copy(output_log_path, f"{run_folder}/alphafold_results/pipeline.log")

            except Exception as e:
                log(f"    ERROR: {e}")
                import traceback
                traceback.print_exc()
                out["plddt"].append(0.0)
                out["ptm"].append(0.0)
                out["iptm"].append(0.0)
                out["pae"].append(31.0)
                if save_incrementally and run_folder:
                    save_incremental_af_results(data, labels, best, loc_dir, run_folder, fasta_lines)

    log("\n" + "=" * 80)
    log("SUMMARY")
    log("=" * 80)
    if best["score"] > 0:
        log(f"\n✓ Best: design {best['design']} seq {best['n']}")
        log(f"  pLDDT: {best['plddt']:.3f}, pTM: {best['ptm']:.3f}, iPTM: {best['iptm']:.3f}, Ranking: {best['score']:.3f}")

df = pd.DataFrame(data, columns=labels)
df = df.sort_values(by="ranking_score", ascending=False)
df.to_csv(f'{loc_dir}/design_validation_results.csv', index=False)

print(f"\n✓ Complete!")
print(f"  Results: {loc_dir}/design_validation_results.csv")

if run_folder:
    af_folder = f"{run_folder}/alphafold_results"
    shutil.copy(f'{loc_dir}/design_validation_results.csv', f'{af_folder}/design_validation_results.csv')
    if os.path.exists(f"{loc_dir}/best.pdb"): shutil.copy(f"{loc_dir}/best.pdb", f"{af_folder}/best.pdb")
    shutil.copy(f"{loc_dir}/design.fasta", f"{af_folder}/validated.fasta")
    shutil.copy(output_log_path, f"{af_folder}/pipeline.log")
    print(f"✓ Final results saved to: {af_folder}")

In [ ]:
#@title Cell 8: Display Best Result
import py3Dmol
import glob
import ipywidgets as widgets
from IPython.display import display

def plot_pdb(num="best"):
  pred_pdb_file = None
  if num == "best":
    best_path = f"{loc_dir}/best.pdb"
    if os.path.exists(best_path):
      pred_pdb_file = best_path
      with open(best_path, "r") as f:
        header = f.readline().split()
        if len(header) >= 6: print(f"Displaying Global Best: Design {header[3]}, Sequence {header[5]}")
    else:
      print(f"Error: {best_path} not found.")
      return
  else:
    search_pattern = f"{loc_dir}/all_pdb/design{num}_n*.pdb"
    found_files = sorted(glob.glob(search_pattern))
    if found_files:
      pred_pdb_file = found_files[0]
      print(f"Displaying Design {num}")
    else:
      print(f"Error: No PDBs found for design {num}")
      return

  with open(pred_pdb_file, 'r') as f: pred_pdb_str = f.read()
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
  view.addModel(pred_pdb_str, 'pdb', {'hbondCutoff': 4.0})
  view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})
  view.zoomTo()
  view.show()

if 'num_designs' in globals() and num_designs > 1:
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(options=["best"] + [str(k) for k in range(num_designs)], value="best", description='Design:')
  dropdown.observe(on_change)
  output = widgets.Output()
  display(widgets.VBox([dropdown, output]))
  with output: plot_pdb(dropdown.value)
else:
  plot_pdb()

In [ ]:
#@title Cell 9: Save/Download Results (Final)
import os
import shutil
from datetime import datetime

zip_name = f"{path}_results.zip"
os.system(f"zip -r {zip_name} {loc_dir} outputs/traj/{path}*")
print(f"✓ Created: {zip_name}")

if use_google_drive:
    if run_folder and os.path.exists(run_folder):
        final_folder = run_folder
    else:
        drive_path = f"/content/drive/MyDrive/{drive_output_folder}"
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_folder = f"{drive_path}/{project_name}_{timestamp}"
        os.makedirs(final_folder, exist_ok=True)

    drive_zip_path = f"{final_folder}/{zip_name}"
    shutil.copy(zip_name, drive_zip_path)
    print(f"✓ Results zip saved to: {drive_zip_path}")

    print(f"\n{'='*60}")
    print(f"✓ ALL RESULTS SAVED TO GOOGLE DRIVE")
    print(f"  Location: {final_folder}")
    print(f"")
    print(f"  📁 Folder structure:")
    print(f"     {final_folder}/")
    print(f"     ├── config.txt                    # Run configuration")
    print(f"     ├── input_{pdb_id}.pdb            # Input structure")
    print(f"     ├── rfdiffusion_backbones/        # Backbone PDBs")
    print(f"     │   └── design_*.pdb")
    print(f"     ├── mpnn_sequences/               # MPNN outputs")
    print(f"     │   ├── mpnn_raw.fasta")
    print(f"     │   ├── mpnn_corrected_for_alphafold.fasta")
    print(f"     │   ├── sequence_metadata.csv")
    print(f"     │   └── colabfold_batch.csv       # ⭐ Ready for HPC!")
    print(f"     ├── alphafold_results/            # AlphaFold outputs")
    print(f"     │   ├── design_validation_results.csv")
    print(f"     │   ├── best.pdb")
    print(f"     │   ├── validated.fasta")
    print(f"     │   └── all_pdb/")
    print(f"     └── {zip_name}       # Complete archive")
    print(f"{'='*60}")
else:
    from google.colab import files
    print("\n⚠️  Browser download requires active connection.")
    print("   For background execution, set use_google_drive = True")
    files.download(zip_name)